# Session 4 — Gradient Descent, Hyperparameters, and Feature Scaling
### Day 1 | FDP: Machine Learning for Materials and Metallurgical Engineering

Session 3 fit Hall-Petch and Cu using `np.polyfit` — a closed-form solver that finds the best (σ0, k) directly. Today we build the **alternative, iterative method** ourselves: gradient descent. We'll use it to understand *why* it needs careful tuning (hyperparameters), and *why* feature scaling matters — using the same small, familiar Hall-Petch dataset throughout so every step is easy to track.

**This session has four parts:**
- **Part A** — Deriving and implementing gradient descent by hand
- **Part B** — Convergence & convexity (why gradient descent is guaranteed to work here)
- **Part C** — Hyperparameters: learning rate, batch size, epochs
- **Part D** — Multiple features & feature scaling

## Setup — Import Libraries and Recall the Hall-Petch Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.array([0.0018, 0.0025, 0.004, 0.007, 0.016, 0.060, 0.25])
sigma_y = np.array([530, 450, 380, 300, 230, 155, 115])

x = d ** -0.5     # the linearized feature, same as Day 1 morning
y = sigma_y
n = len(x)

# Ground truth, from Day 1 morning's np.polyfit
k_true, sigma0_true = np.polyfit(x, y, 1)
print(f"Ground truth (np.polyfit): sigma_0 = {sigma0_true:.2f}, k = {k_true:.2f}")
print(f"Feature range (d^-0.5): {x.min():.2f} to {x.max():.2f}")


---
## Part A — Deriving Gradient Descent by Hand

### The Cost Function

We measure fit quality the same way as Session 3 — mean squared error — but now written as a function of the two parameters we're searching for, $\sigma_0$ and $k$:

$$J(\sigma_0, k) = \frac{1}{n}\sum_{i=1}^{n} \left(y_i - (\sigma_0 + k\,x_i)\right)^2$$

### The Update Rule

Gradient descent repeatedly nudges $\sigma_0$ and $k$ in the direction that reduces $J$ the fastest — found by taking the partial derivative of $J$ with respect to each parameter:

$$\frac{\partial J}{\partial k} = -\frac{2}{n}\sum x_i\,(y_i - \hat{y}_i) \qquad\qquad \frac{\partial J}{\partial \sigma_0} = -\frac{2}{n}\sum (y_i - \hat{y}_i)$$

Then both parameters are updated **simultaneously**, scaled by a small step size (the *learning rate*, $\alpha$):

$$k \leftarrow k - \alpha \frac{\partial J}{\partial k} \qquad\qquad \sigma_0 \leftarrow \sigma_0 - \alpha \frac{\partial J}{\partial \sigma_0}$$

### Implementing It

In [ ]:
def gradient_descent(x, y, lr, n_iterations, track_every=None):
    n = len(x)
    k_, sigma0_ = 0.0, 0.0
    loss_history = []
    snapshots = []

    for i in range(n_iterations):
        y_pred = sigma0_ + k_ * x
        loss = np.mean((y - y_pred) ** 2)
        loss_history.append(loss)

        if track_every and i % track_every == 0:
            snapshots.append((i, sigma0_, k_, loss))

        dk = (-2/n) * np.sum(x * (y - y_pred))
        dsigma0 = (-2/n) * np.sum(y - y_pred)
        k_ -= lr * dk
        sigma0_ -= lr * dsigma0

    return sigma0_, k_, loss_history, snapshots


In [ ]:
sigma0_fit, k_fit, losses, snapshots = gradient_descent(x, y, lr=0.001, n_iterations=100001, track_every=20000)

print(f"{'Iteration':<12}{'sigma_0':<12}{'k':<10}{'Loss':<12}")
for it, s0, k1, l in snapshots:
    print(f"{it:<12}{s0:<12.2f}{k1:<10.2f}{l:<12.2f}")

print()
print(f"Final result:  sigma_0 = {sigma0_fit:.2f}, k = {k_fit:.2f}")
print(f"np.polyfit:    sigma_0 = {sigma0_true:.2f}, k = {k_true:.2f}")


### Watching It Converge

In [ ]:
plt.figure(figsize=(6.5, 4.5))
plt.plot(losses[:20000])
plt.xlabel("Iteration")
plt.ylabel("Loss (MSE)")
plt.title("Gradient descent loss curve (lr = 0.001)")
plt.tight_layout()
plt.show()


---
## Quick Check A

Why do we update $\sigma_0$ and $k$ **simultaneously** (using their old values to compute both new values) rather than updating $k$ first and then using the *new* $k$ to update $\sigma_0$?

**(i)** It doesn't actually matter which order you use
**(ii)** Updating sequentially would mean the second update no longer follows the true gradient direction of the original point
**(iii)** Simultaneous updates are simply faster to compute

*Think about it, then check the answer below.*

**Answer: (ii)** — the gradient tells us the direction to move from a *specific* point $(\sigma_0, k)$. If we update $k$ first and then compute $\sigma_0$'s gradient using the *new* $k$, we're no longer moving in the direction the original gradient actually pointed – the math only guarantees improvement if both updates come from the same starting point.

---
## Part B — Convergence & Convexity

The loss curve above is smooth and monotonically decreasing — no bumps, no getting stuck partway. This isn't a coincidence: for linear regression, the loss function $J(\sigma_0, k)$ is always **convex** — shaped like a smooth bowl with a single global minimum, no matter what data you use.

This guarantees that gradient descent, started from *any* point, will always find its way to the bottom — the true best fit. Let's see the bowl shape directly.

In [ ]:
sigma0_range = np.linspace(0, 160, 60)
k_range = np.linspace(-10, 50, 60)
S0, K = np.meshgrid(sigma0_range, k_range)

loss_surface = np.zeros_like(S0)
for i in range(S0.shape[0]):
    for j in range(S0.shape[1]):
        y_pred = S0[i, j] + K[i, j] * x
        loss_surface[i, j] = np.mean((y - y_pred) ** 2)

fig = plt.figure(figsize=(13, 5.5))

# Left panel: 3D bowl
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(S0, K, loss_surface, cmap="viridis", alpha=0.85, linewidth=0, antialiased=True)
ax1.scatter([sigma0_true], [k_true], [loss_surface.min()], color="red", s=60, marker="*")
ax1.set_xlabel("sigma_0")
ax1.set_ylabel("k")
ax1.set_zlabel("Loss (MSE)")
ax1.set_title("The loss surface is a single smooth bowl (convex)")
ax1.view_init(elev=28, azim=-60)

# Right panel: 2D contour (top-down view of the same bowl)
ax2 = fig.add_subplot(1, 2, 2)
contour = ax2.contourf(S0, K, loss_surface, levels=30, cmap="viridis")
plt.colorbar(contour, ax=ax2, label="Loss (MSE)")
ax2.scatter([sigma0_true], [k_true], color="red", marker="*", s=200, label="True minimum (np.polyfit)")
ax2.set_xlabel("sigma_0")
ax2.set_ylabel("k")
ax2.set_title("Same bowl, viewed from directly above")
ax2.legend()

plt.tight_layout()
plt.show()

**Why this matters going forward:** this convexity guarantee is specific to linear regression. Later this week (Day 4, neural networks), the loss surface is *not* convex — it can have multiple local minima, flat regions, and other complications. That's part of why training a neural network is harder to reason about than fitting a line.

### Watching Gradient Descent Walk Down the Bowl

The lecture slides showed this as a frame-by-frame animation — the point starts high on the bowl's wall and takes one step at a time toward the bottom. We can't animate a static notebook the same way, but we can plot the **entire path** it takes, which shows the same thing at a glance: gradient descent doesn't jump straight to the minimum, it takes a curving route determined by the local slope at every point along the way.

In [ ]:
# Re-run with much finer tracking, purely to get a smooth visible path
_, _, _, path_snapshots = gradient_descent(x, y, lr=0.001, n_iterations=20000, track_every=100)

path_sigma0 = [s[1] for s in path_snapshots]
path_k      = [s[2] for s in path_snapshots]
path_loss   = [s[3] for s in path_snapshots]

fig = plt.figure(figsize=(13, 5.5))

# Left: path on the 3D bowl
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(S0, K, loss_surface, cmap="viridis", alpha=0.5, linewidth=0)
ax1.plot(path_sigma0, path_k, path_loss, color="black", linewidth=1.5, marker="o", markersize=3)
ax1.scatter([path_sigma0[0]], [path_k[0]], [path_loss[0]], color="blue", s=80, label="Start")
ax1.scatter([sigma0_true], [k_true], [loss_surface.min()], color="red", s=80, marker="*", label="Minimum")
ax1.set_xlabel("sigma_0")
ax1.set_ylabel("k")
ax1.set_zlabel("Loss (MSE)")
ax1.set_title("Gradient descent's path down the bowl")
ax1.view_init(elev=28, azim=-60)
ax1.legend()

# Right: same path, top-down, colored by how far along we are
ax2 = fig.add_subplot(1, 2, 2)
ax2.contour(S0, K, loss_surface, levels=25, cmap="viridis", alpha=0.6)
sc = ax2.scatter(path_sigma0, path_k, c=range(len(path_sigma0)), cmap="autumn_r", s=25, zorder=3)
ax2.plot(path_sigma0, path_k, color="black", linewidth=0.8, zorder=2)
plt.colorbar(sc, ax=ax2, label="Iteration (0 -> 20000)")
ax2.scatter([sigma0_true], [k_true], color="blue", marker="*", s=200, label="Minimum", zorder=4)
ax2.set_xlabel("sigma_0")
ax2.set_ylabel("k")
ax2.set_title("Same path, viewed from above")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Path started at (sigma_0={path_sigma0[0]:.1f}, k={path_k[0]:.1f}), loss={path_loss[0]:.1f}")
print(f"Path ended at   (sigma_0={path_sigma0[-1]:.1f}, k={path_k[-1]:.1f}), loss={path_loss[-1]:.1f}")

Notice the path curves rather than moving in a straight line to the minimum — early on, the gradient points toward the steepest immediate direction, which isn't necessarily aimed straight at the minimum, especially when the bowl is stretched (as it is here). This same curving behavior is a big part of *why* the learning rate matters so much, which is exactly where we're headed next.

---
## Quick Check B

If linear regression's loss surface has a single global minimum, does the *starting point* we choose for $(\sigma_0, k)$ matter?

**(i)** Yes, a bad starting point means gradient descent might never find the true minimum
**(ii)** No, gradient descent will reach the same minimum regardless of starting point (assuming a suitable learning rate)
**(iii)** It matters only if the starting point is negative

*Think about it, then check the answer below.*

**Answer: (ii)** — thanks to convexity, any reasonable starting point eventually reaches the same global minimum. This is *not* true in general for non-convex problems, where a poor starting point can get stuck in a local minimum instead.

---
## Part C — Hyperparameters: Learning Rate, Batch Size, and Epochs

**Hyperparameters** are settings we choose *before* training – unlike $\sigma_0$ and $k$, they aren't learned from data. The learning rate is the one we already know matters a lot: recall our own notebook from Day 1 morning, where lr=0.01 caused the fit to diverge. Let's see that failure happen live, side by side with a learning rate that works.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # the diverging run below intentionally overflows

_, _, losses_diverge, _ = gradient_descent(x, y, lr=0.01, n_iterations=8)
_, _, losses_converge, _ = gradient_descent(x, y, lr=0.001, n_iterations=8)

print("lr = 0.01  (too large):")
for i, l in enumerate(losses_diverge):
    print(f"  iteration {i}: loss = {l:,.1f}")

print()
print("lr = 0.001 (works):")
for i, l in enumerate(losses_converge):
    print(f"  iteration {i}: loss = {l:.2f}")


Notice: with lr=0.01, the loss doesn't just fail to improve — it grows *explosively*, roughly 10x larger each iteration. This is what "too high a learning rate" actually looks like in practice, not just in theory.

### A Closer Look: Learning Rate on Real, Larger Data

The comparison above used just 8 iterations on our tiny 7-point Hall-Petch set — enough to see the idea, but not really a *study* of the effect. Let's do this properly: sweep across a range of learning rates on the same 198-point real Cu dataset from Day 1 Session 3, and watch the full story play out — too slow, good, and diverging — side by side.

In [ ]:
import pandas as pd

cu_df = pd.read_csv("https://raw.githubusercontent.com/vijindal/fdp-ml/main/notebooks/hallpetch_multimetal_data.csv")
cu_df = cu_df[cu_df['element'] == 'Cu']

x_cu = (cu_df['grain_size_um'] ** -0.5).values
y_cu = cu_df['flow_stress_MPa'].values

print(f"Cu dataset: {len(x_cu)} real measurements (vs. our 7-point toy set)")
print(f"Feature (d^-0.5) range: {x_cu.min():.2f} to {x_cu.max():.2f}")

In [ ]:
learning_rates = [0.0001, 0.001, 0.01, 0.03]
n_iter_sweep = 200

plt.figure(figsize=(7, 5))
for lr in learning_rates:
    _, _, losses_lr, _ = gradient_descent(x_cu, y_cu, lr=lr, n_iterations=n_iter_sweep)
    plt.plot(losses_lr, label=f"lr = {lr}")

plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Loss (MSE, log scale)")
plt.title("Learning Rate Sweep on Real Cu Data (198 points)")
plt.legend()
plt.tight_layout()
plt.show()

**All four tell a different part of the story, at the same 200 iterations:**

- **lr = 0.0001** — barely moved (loss ≈ 52,000, still far from the ≈24,300 optimum) — too slow to be practical
- **lr = 0.001** — real, steady progress, but not yet converged (loss ≈ 32,000)
- **lr = 0.01** — essentially converged already (loss ≈ 24,500, very close to optimal) — the sweet spot here
- **lr = 0.03** — diverges catastrophically (loss explodes past 10¹⁴) — visibly off the top of the log-scale plot within a handful of iterations

This is the same pattern we saw on the tiny dataset, but now with a real, larger, noisier dataset — and a genuine sweep instead of a single before/after comparison. Note that the *right* learning rate is always dataset- and feature-scale-dependent — this is exactly why feature scaling (Part D, coming up) matters so much: it changes what "too large" even means.

---
## Quick Check C2

On the log-scale loss plot above, lr = 0.03's curve shoots upward almost immediately, while lr = 0.0001's curve is nearly flat. What does a *rising* loss curve indicate, as opposed to one that's simply flat?

**(i)  Both indicate the same problem — the learning rate is wrong either way**
**(ii)  A flat curve means the steps are too small to make visible progress; a rising curve means the steps are so large they overshoot the minimum and move further away each iteration — a qualitatively different failure mode**
**(iii)  A rising loss curve means the data itself is bad**

*Think about it, then check the answer below.*

**Answer: (ii)** — too-small and too-large learning rates fail for different reasons. Too small simply wastes iterations without making meaningful progress; too large actively overshoots the minimum on every step, moving further away rather than closer, and the loss can grow without bound rather than plateau.

### Batch Size and Epochs

Two more hyperparameters worth knowing:

- **Batch size** – how many data points are used to compute each gradient step. With large datasets, using all points every step (*full-batch*, what we've done here) can be slow, so mini-batches (a random subset each step) or a single point at a time (*stochastic gradient descent*, SGD) are common alternatives.
- **Epochs** – one epoch means the model has seen every training example once. A full-batch update happens once per epoch; mini-batch and SGD update many times per epoch instead – exactly what drives the differences we're about to see.

Even at our data scale (198 points), the mechanics – and the noise mini-batching and SGD introduce – are identical to what you'd see at production scale. Let's see it directly on the same real Cu dataset we just used above.

---
## Quick Check C

Based on what we just saw, what's the safest way to choose a learning rate for a new problem?

**(i)** Always use the largest learning rate possible, for the fastest convergence
**(ii)** Start small and only increase it if convergence seems too slow, watching the loss curve for divergence
**(iii)** The learning rate doesn't matter as long as you run enough iterations

*Think about it, then check the answer below.*

**Answer: (ii)** — as we just saw directly, too large a learning rate causes the loss to explode rather than converge, regardless of how many iterations you run. Starting conservatively and watching the loss curve is the standard practical approach.

In [ ]:
def gradient_descent_batched(x, y, lr, n_epochs, batch_size, seed=42):
    """Same update rule as gradient_descent(), but processes data in batches per epoch
    instead of using all points at once every iteration."""
    rng = np.random.default_rng(seed)
    n = len(x)
    k_, sigma0_ = 0.0, 0.0
    loss_history = []

    for epoch in range(n_epochs):
        idx = rng.permutation(n)  # shuffle each epoch
        x_shuf, y_shuf = x[idx], y[idx]

        for start in range(0, n, batch_size):
            xb = x_shuf[start:start + batch_size]
            yb = y_shuf[start:start + batch_size]
            y_pred = sigma0_ + k_ * xb
            dk = (-2 / len(xb)) * np.sum(xb * (yb - y_pred))
            dsigma0 = (-2 / len(xb)) * np.sum(yb - y_pred)
            k_ -= lr * dk
            sigma0_ -= lr * dsigma0

        # track loss on the FULL dataset at the end of each epoch, for fair comparison
        y_pred_full = sigma0_ + k_ * x
        loss_history.append(np.mean((y - y_pred_full) ** 2))

    return sigma0_, k_, loss_history

In [ ]:
batch_configs = [("Full-batch (n=198)", 198), ("Mini-batch (n=32)", 32), ("SGD (n=1)", 1)]
n_epochs_demo = 25

plt.figure(figsize=(7, 5))
for label, bs in batch_configs:
    s0, k, losses_bs = gradient_descent_batched(x_cu, y_cu, lr=0.01, n_epochs=n_epochs_demo, batch_size=bs)
    plt.plot(range(1, n_epochs_demo + 1), losses_bs, marker='o', markersize=3, label=label)
    updates_per_epoch = int(np.ceil(len(x_cu) / bs))
    print(f"{label:<22} updates/epoch = {updates_per_epoch:<4} final loss = {losses_bs[-1]:>10,.0f}  "
          f"sigma_0 = {s0:.2f}, k = {k:.2f}")

plt.xlabel("Epoch")
plt.ylabel("Loss (MSE), measured on full data at end of each epoch")
plt.title("Full-Batch vs. Mini-Batch vs. SGD on Real Cu Data")
plt.legend()
plt.tight_layout()
plt.show()

**The update-count difference drives everything here:** full-batch gets exactly 1 update per epoch, mini-batch(32) gets ⌈198/32⌉ = 7, and SGD gets all 198 – far more frequent updates for the same number of epochs.

- **Full-batch** – perfectly smooth, monotonically decreasing curve, as guaranteed by convexity (Part B) – but the *slowest* of the three here, since it only updates once per epoch
- **Mini-batch (32)** – visibly noisier epoch-to-epoch, but reaches a lower loss faster overall, thanks to its extra updates
- **SGD (1)** – by far the noisiest – occasional dramatic spikes where a single unusual point pulls the parameters off course – yet it still often reaches competitive loss values, because there are so many more update opportunities per epoch

This is exactly the "noise vs. speed" tradeoff the theory describes – more frequent, noisier updates can reach a good answer faster in wall-clock terms, at the cost of a bumpier ride.

---
## Quick Check — Batch Size

Why does SGD's loss curve show occasional huge spikes that full-batch gradient descent never does?

**(i)  SGD has a bug and shouldn't be used**
**(ii)  SGD updates using a single, randomly-chosen data point each step – an unusual or noisy point can pull the parameters strongly in the wrong direction for that one step, unlike full-batch which always averages over every point**
**(iii)  The spikes mean the learning rate is too high**

*Think about it, then check the answer below.*

**Answer: (ii)** – full-batch gradient descent always averages the gradient over all 198 points, so any one unusual point's influence is diluted. SGD computes each update from a single point, so an outlier or unusually-placed point can dominate that one step entirely – producing the sharp spikes visible above. This is the structural cost of SGD's extra update frequency, not a sign anything is wrong with the learning rate itself.

---
## Part C (continued) — The Same Hyperparameters, in a Real ML Library

Everything above was gradient descent we wrote ourselves, so we could see exactly what "learning rate" and "batch size" actually control. But you'll almost never write gradient descent by hand in practice — you'll use a library. The good news: **it's the exact same knobs.**

`scikit-learn`'s `SGDRegressor` fits a linear model using gradient descent, and exposes learning rate as a hyperparameter directly (`eta0`). Let's repeat the learning-rate sweep from earlier — same real Cu data, same idea — using a production library instead of our own code, and confirm we see the same story.

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

# SGDRegressor expects 2D input for X
X_cu = x_cu.reshape(-1, 1)

# Same learning rates we tested by hand above
eta0_values = [0.0001, 0.001, 0.01, 0.03]
n_epochs_sweep = 200

plt.figure(figsize=(7, 5))
for eta0 in eta0_values:
    model = SGDRegressor(
        learning_rate="constant",  # keep the learning rate fixed, same as our own code
        eta0=eta0,
        max_iter=1,                # we control iteration count ourselves, one epoch at a time
        warm_start=True,           # keep learned weights between .fit() calls instead of resetting
        random_state=42,
    )
    epoch_losses = []
    for epoch in range(n_epochs_sweep):
        model.fit(X_cu, y_cu)
        y_pred = model.predict(X_cu)
        loss = ((y_cu - y_pred) ** 2).mean()
        epoch_losses.append(loss)

    plt.plot(epoch_losses, label=f"eta0 = {eta0}")

plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE, log scale)")
plt.title("Same Learning-Rate Sweep, via scikit-learn's SGDRegressor")
plt.legend()
plt.tight_layout()
plt.show()

**Compare this to the sweep a few cells back, on our own hand-written gradient descent.** Same shape, same failure mode at the high learning rate, same story: too small crawls, too large explodes. `SGDRegressor` isn't doing anything conceptually different from the code we wrote ourselves — it's the identical algorithm, just optimized and production-ready.

One real difference worth knowing: `SGDRegressor` is *not* the tool you'd reach for to just "fit a line" in practice — for plain linear regression, `sklearn.linear_model.LinearRegression` solves it directly (the same closed-form, SVD-based approach as `np.polyfit`, no learning rate needed at all — see this morning's discussion of why `np.polyfit` isn't gradient descent). `SGDRegressor` matters for the cases where a direct solution *isn't* available — very large datasets that don't fit in memory at once, or, as we'll see across the rest of the week, models far more complex than a straight line.

---
## Quick Check C3

If `LinearRegression` in scikit-learn already solves for the exact best-fit line directly (no learning rate, no iterations), why would anyone use `SGDRegressor` instead?

(i) `SGDRegressor` is always more accurate
(ii) `SGDRegressor` scales to datasets too large to fit in memory at once, since it updates from small batches rather than needing the whole dataset simultaneously
(iii) `SGDRegressor` doesn't require any hyperparameters, making it simpler to use

**Answer: (ii)** — `LinearRegression`'s direct solve needs the full dataset in memory at once (like `np.polyfit`). `SGDRegressor` processes data incrementally, which is what makes gradient descent the method of choice once a dataset is too large for a direct solve, or once the model is too complex to have one at all (as with neural networks, later this week). It's not about accuracy — for a problem this simple, both reach essentially the same answer — and it clearly does require hyperparameters (we just tuned one above).

---
## Part D — Multiple Features & Feature Scaling

**Multiple features:** so far we've had one feature ($d^{-1/2}$). With several features (as we'll see from Day 2 onward, once we're using multiple materials descriptors at once), the same equation simply extends:

$$y' = b + w_1 x_1 + w_2 x_2 + \dots + w_n x_n$$

and gradient descent works exactly the same way — one partial derivative per weight, all updated simultaneously.

### Feature Scaling

Here's something worth connecting back to our own experience today: our Hall-Petch feature, $d^{-1/2}$, ranges from about 2 to 23.6 — **unscaled**. This is exactly *why* we needed such a small learning rate (0.001) to avoid diverging. Let's confirm this directly: if we scale the feature down first, does a much larger learning rate suddenly become safe?

In [ ]:
x_scaled = x / x.max()
print(f"Original feature range:  {x.min():.2f} to {x.max():.2f}")
print(f"Scaled feature range:    {x_scaled.min():.2f} to {x_scaled.max():.2f}")

sigma0_scaled, k_scaled, losses_scaled, _ = gradient_descent(x_scaled, y, lr=0.5, n_iterations=5000)

# Unscale k back to the original units (sigma_0 is unaffected by scaling x)
k_unscaled = k_scaled / x.max()

print()
print(f"Using SCALED features with lr=0.5 (500x larger than before!), only 5000 iterations:")
print(f"  sigma_0 = {sigma0_scaled:.2f}, k (unscaled) = {k_unscaled:.2f}")
print(f"  Compare to np.polyfit:  sigma_0 = {sigma0_true:.2f}, k = {k_true:.2f}")


Scaling the feature let us use a learning rate **500 times larger**, converging in a fraction of the iterations – and still landing on the exact same answer. This is the practical reason feature scaling is considered standard practice: it makes gradient descent faster and far less sensitive to learning-rate choice.

---
## Quick Check D

Why did scaling the feature allow such a dramatically larger learning rate?

**(i)** Scaling changes the true relationship between grain size and yield stress
**(ii)** With a smaller feature range, a given learning rate produces smaller, safer parameter updates – so a larger rate can be used without overshooting
**(iii)** Scaling has no real effect; the result was a coincidence

*Think about it, then check the answer below.*

**Answer: (ii)** — the size of each gradient descent step depends on the feature's scale. A feature ranging up to 23.6 produces much larger gradient values (and so much larger steps) than one scaled down to a maximum of 1.0. Scaling levels the playing field, which is why it's standard practice whenever features have very different natural ranges — especially relevant from Day 2 onward, when we'll have several descriptors with different units and scales at once.

## Wrap-Up

- Built gradient descent from scratch and confirmed it converges to the exact same answer as `np.polyfit`
- Saw *why* it's guaranteed to work for linear regression: a convex, single-minimum loss surface
- Reproduced our own real learning-rate divergence bug live, and used it to motivate careful hyperparameter choice
- Confirmed, numerically, that feature scaling allows a dramatically larger learning rate — directly explaining why our original Hall-Petch gradient descent needed such a small learning rate in the first place

**Next:** Day 1 closes with the capstone project briefing — choose a problem, and think about which of this week's techniques might apply to it.